In [1]:
import pathlib
import cytotable
from ome_arrow import OMEArrow

In [2]:
bandicoot = (
    pathlib.Path("~/mnt/bandicoot").expanduser().resolve()
    / "PCCMA_data"
    / "SK-N-AS_repo1_screen"
)

In [3]:
sqlite_path = bandicoot / "SQLite_outputs"
screen_folders = [
    bandicoot / "Original REPO1 Screen",
    bandicoot / "REPO1 Row O Repeat",
]

In [4]:
warehouse_path = pathlib.Path("/mnt/storage_18tb/sknas_warehouse")
warehouse_path.mkdir(parents=True, exist_ok=True)

In [5]:
# Build a lookup: plate_id -> Images path, searching only 1 level deep
print("Building image directory index...")
image_index = {}
for screen_folder in screen_folders:
    for measurement_dir in screen_folder.iterdir():
        if not measurement_dir.is_dir():
            continue
        # Extract plate ID from folder name (e.g. BR00148926)
        parts = measurement_dir.name.split("__")
        if parts:
            plate_id = parts[0]
            image_dir = measurement_dir / "Images"
            if image_dir.exists():
                image_index[plate_id] = image_dir

print(f"Found images for {len(image_index)} plates: {list(image_index.keys())}")

# Find all plate SQLite folders
plate_dirs = sorted([p for p in sqlite_path.iterdir() if p.is_dir()])
print(f"Found {len(plate_dirs)} SQLite plates")

Building image directory index...
Found images for 22 plates: ['BR00148926', 'BR00148938', 'BR00148936', 'BR00148930', 'BR00148944', 'BR00148939', 'BR00148940', 'BR00148942', 'BR00148932', 'BR00148937', 'BR00148934', 'BR00148931', 'BR00148933', 'BR00148935', 'BR00148929', 'BR00148943', 'BR00148928', 'BR00148941', 'BR00148927', 'BR00148945', 'BR00149048', 'BR00149049']
Found 29 SQLite plates


In [ ]:
successes = []
failures = []

for plate_dir in plate_dirs:
    sqlite_file = plate_dir / "alsf_morphology_features.sqlite"

    if not sqlite_file.exists():
        print(f"  Skipping {plate_dir.name} — no sqlite file found")
        continue

    plate_warehouse = warehouse_path / plate_dir.name

    if plate_warehouse.exists() and any(plate_warehouse.iterdir()):
        print(f"  Skipping {plate_dir.name} — already completed")
        successes.append(plate_dir.name)
        continue

    image_dir = image_index.get(plate_dir.name)

    print(f"\n{'='*60}")
    print(f"Plate:   {plate_dir.name}")
    print(f"SQLite:  {sqlite_file}")
    print(f"Images:  {image_dir or 'NOT FOUND'}")
    print(f"{'='*60}")

    try:
        result = cytotable.convert(
            source_path=str(sqlite_file),
            source_datatype="sqlite",
            dest_path=str(plate_warehouse),
            dest_backend="iceberg",
            dest_datatype="parquet",
            preset="cellprofiler_sqlite",
            image_dir=str(image_dir) if image_dir else None,
        )
        print(f"✓ Done: {plate_dir.name} → {result}")
        successes.append(plate_dir.name)

    except Exception as e:
        print(f"✗ Failed: {plate_dir.name} — {e}")
        failures.append((plate_dir.name, str(e)))

print(f"\n\n{'='*60}")
print(f"COMPLETE — {len(successes)} succeeded, {len(failures)} failed")
if failures:
    print("\nFailed plates:")
    for name, err in failures:
        print(f"  ✗ {name}: {err}")
print(f"\nWarehouse: {warehouse_path}")


  Skipping BR00148919 — already completed
  Skipping BR00148920 — already completed

Plate:   BR00148921
SQLite:  /home/juliacurd/mnt/bandicoot/PCCMA_data/SK-N-AS_repo1_screen/SQLite_outputs/BR00148921/alsf_morphology_features.sqlite
Images:  NOT FOUND


In [ ]:
# Read tables
joined_profiles = cytotable.read_table(
    warehouse_path,
    "joined_profiles"
)

print("\nJoined profiles:")
print(joined_profiles[["Metadata_ObjectID"]].head())

In [ ]:
image_crops = cytotable.read_table(
    warehouse_path,
    "image_crops"
)

print("\nImage crops:")
print(
    image_crops[
        [
            "Metadata_ObjectID",
            "Metadata_ImageCropID",
            "source_image_file",
        ]
    ].head()
)

In [ ]:
source_images = cytotable.read_table(
    warehouse_path,
    "source_images"
)

print("\nSource images:")
print(
    source_images[
        [
            "Metadata_ImageID",
            "source_image_file",
        ]
    ].head()
)

In [ ]:
profile_with_images = cytotable.read_table(
    warehouse_path,
    "profile_with_images"
)

print("\nProfile with images:")
print(
    profile_with_images[
        ["Metadata_ObjectID", "source_image_file"]
    ].head()
)